In [2]:
# positional encoding 

import torch 
import torch.nn as nn 
import torch.nn.functional as F


In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self , max_length , emb_dim , dropout = 0.1):
        super().__init__()
        self.pos_emb = nn.Parameter(torch.randn(max_length , emb_dim) *0.02)
        self.dropout = nn.Dropout(dropout)
    def forward(self , x):
        x = x + self.pos_emb[:x.size(1)]
        return self.dropout(x)


In [ ]:
class MultiheadAttention(nn.Module):
    def __init__(self , emb_dim , num_heads , dropout = 0.1):
        super().__init__()
        self.h = num_heads
        self.d = emb_dim // num_heads
        self.q_proj = nn.Linear(emb_dim , emb_dim)
        self.k_proj = nn.Linear(emb_dim , emb_dim)
        self.v_proj = nn.Linear(emb_dim , emb_dim)
        self.out_proj = nn.Linear(emb_dim , emb_dim)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self , x):
        return x.view(x.size(0) , x.size(1) , self.h , self.d).transpose(1 , 2)
    def forward(self , query , key , value , attn_mask = None , key_padding_mask = None):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        scores = q @ k.transpose(2, 3) / (self.d ** 0.5)
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask , -torch.inf)
        if key_padding_mask is not None:
            mask = mask.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2) )
            scores = scores.masked_fill(mask , -torch.inf)
        weights = F.softmax(scores , dim = -1)
        z= self.dropout(weights) @ v
        z = z.transpose(1 , 2)
        z= z.reshape(z.size(0) , z.size(1) , self.h * self.d)
        return (self.out_proj(z),weights)

      

tensor([[[-0.4841, -1.6923,  0.9413,  0.0492, -1.5715, -0.9146, -2.2491,
          -0.9519],
         [-0.3724,  0.0554,  1.8274, -0.5775,  0.4222,  0.6731, -2.1182,
           1.1578],
         [-1.0082, -0.2787, -0.6561, -0.5064, -0.5628,  1.8757,  0.6745,
          -1.0956],
         [-0.5186, -0.0777, -0.4980,  0.1328,  0.3272,  0.2419,  1.2880,
          -2.0731],
         [-2.1568,  0.9182, -0.1122,  0.5021,  0.1076, -0.0984,  0.5420,
          -0.2304]],

        [[-0.7014,  0.3928,  0.1698,  1.2220, -0.9697, -1.1083,  0.3024,
          -1.2720],
         [-0.1261,  1.1411,  2.0876,  0.7609,  0.5749, -0.4492,  0.4238,
           0.5121],
         [ 0.1077, -1.0680, -1.4095,  1.0075, -0.1682, -0.1901, -0.2257,
           0.0984],
         [-0.6370, -1.1716,  0.6378,  1.3207,  0.7203,  0.7499,  0.2507,
          -1.0179],
         [-0.4567,  1.4626,  1.6782, -0.1620, -0.3145, -0.4823, -1.3785,
           1.7548]]])